In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [3]:
df = sns.load_dataset('iris')

In [4]:
df

,sepal_length,sepal_width,petal_length,petal_width,species
0,5.1,3.5,1.4,0.2,setosa
1,4.9,3.0,1.4,0.2,setosa
2,4.7,3.2,1.3,0.2,setosa
3,4.6,3.1,1.5,0.2,setosa
4,5.0,3.6,1.4,0.2,setosa
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,virginica
146,6.3,2.5,5.0,1.9,virginica
147,6.5,3.0,5.2,2.0,virginica
148,6.2,3.4,5.4,2.3,virginica


In [6]:
print(df['species'].unique())

['setosa' 'versicolor' 'virginica']


In [7]:
X = df.drop('species' , axis = 1)
y = df['species']

In [19]:
from sklearn.model_selection import train_test_split

# knn model

In [9]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [11]:
from sklearn.neighbors import KNeighborsClassifier

In [15]:
model = KNeighborsClassifier(n_neighbors = 5)

In [16]:
model.fit(X_train,y_train)

,n_neighbors,5
,weights,'uniform'
,algorithm,'auto'
,leaf_size,30
,p,2
,metric,'minkowski'
,metric_params,None
,n_jobs,None


In [18]:
model.score(X_test,y_test)

1.0

# svm model

In [21]:
from sklearn.svm import SVC

In [32]:
model_svm = SVC (gamma = 'auto' )

In [33]:
model_svm.fit(X_train,y_train)

,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'auto'
,coef0,0.0
,shrinking,True
,probability,False
,tol,0.001
,cache_size,200
,class_weight,None
,verbose,False


In [34]:
model_svm.score(X_test,y_test)

1.0

In [35]:
from sklearn.model_selection import GridSearchCV

In [37]:
classifier = GridSearchCV((model_svm) ,{
    'C' : [5,10,15,20,25,30],
    'kernel' : ['linear' , 'rbf' , 'poly' , 'sigmoid']   
}, cv = 5 , return_train_score = False)

**When you use GridSearchCV, you don't need to manually split your data into train and test sets beforehand.**

**GridSearchCV splits the data for you automatically behind the scenes.**


**Here is how it works:**
**It performs Cross-Validation automatically :**
**If you look closely at the very top line of code , you can see cv=5.**
**This tells GridSearchCV to take that plain X and y you passed in, chop it into 5 equal parts (folds), and rotate through them.**

In [51]:
classifier.fit(X_train,y_train)

,estimator,SVC(gamma='auto')
,param_grid,"{'C': [5, 10, ...], 'kernel': ['linear', 'rbf', ...]}"
,scoring,None
,n_jobs,None
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,C,5


In [52]:
results = pd.DataFrame(classifier.cv_results_)

In [53]:
results.head(3)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.001991,0.000843,0.001178,0.000198,5,linear,"{'C': 5, 'kernel': 'linear'}",1.000000,1.000000,0.833333,1.000000,1.000000,0.966667,0.066667,1
1,0.001375,0.000102,0.000973,0.000091,5,rbf,"{'C': 5, 'kernel': 'rbf'}",1.000000,0.958333,0.833333,1.000000,0.958333,0.950000,0.061237,3
2,0.002991,0.001396,0.000987,0.000073,5,poly,"{'C': 5, 'kernel': 'poly'}",0.958333,1.000000,0.833333,0.958333,0.958333,0.941667,0.056519,10


In [59]:
results[['param_C','param_kernel','mean_test_score']].max()

param_C                  30
param_kernel        sigmoid
mean_test_score    0.966667
dtype: object

In [64]:
X = df.drop('species' , axis = 1)
y = df['species']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [70]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler

In [71]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)  

In [72]:
param_grid = {
    'n_neighbors': [3, 5, 7, 9],
    'weights'    : ['uniform', 'distance'],
    'metric'     : ['euclidean', 'manhattan']
}

# 5. GridSearchCV (cv=5 → 4x2x2=16 combos × 5 folds = 80 fits)
gs = GridSearchCV(
    estimator  = KNeighborsClassifier(),
    param_grid = param_grid,
    cv         = 5,
    scoring    = 'accuracy',
    n_jobs     = -1,          # use all CPU cores
    verbose    = 1
)


In [73]:
gs.fit(X_train, y_train)      # CV happens only inside X_train

# 6. Results
print("Best params :", gs.best_params_)
print("Best CV acc :", round(gs.best_score_, 4))

# 7. Final evaluation on the untouched test set
test_acc = gs.score(X_test, y_test)
print("Test acc    :", round(test_acc, 4))

Fitting 5 folds for each of 16 candidates, totalling 80 fits
Best params : {'metric': 'euclidean', 'n_neighbors': 9, 'weights': 'distance'}
Best CV acc : 0.9583
Test acc    : 1.0
